# Comparing RMP 1&2 result document classification
We will compare the manual result document (documents uploaded in the sections result table, result document and other documents) classfication between the two reviewers CP and PR.

First we import `pandas`:

In [2]:
import pandas as pd

We will now read the document classificationd data.

In [3]:
document_CP = pd.read_excel('../../../data/ema_rwd/rmp1&2_documents_manual_CP.xlsx', index_col='path').sort_index()
document_PR = pd.read_excel('../../../data/ema_rwd//rmp1&2_documents_manual_PR.xlsx', index_col='path').sort_index()

display(document_CP.head())
display(document_PR.head())

We will perform a quick check to ensure, that there were no accidental changes made to the template.

In [4]:
check_if_templates_same = document_CP.drop('manual_document_type', axis='columns').compare(
    document_PR.drop('manual_document_type', axis='columns')
).empty

print(f'{check_if_templates_same=}')

Now we will print the unique manually assigned values for both reviewers:

In [5]:
document_CP.manual_document_type.value_counts()

In [6]:
document_PR.manual_document_type.value_counts()

We can now create a replacement map to compare the two documents and compare the Dataframes:

In [7]:
replace_map = {
    'manual_document_type': {
        'abstract of final study report and addendum reports': 'abstract of final study report',
        'abstract of final study report addendum': 'abstract of final study report',
        'poster': 'abstract of final study report',
        'results': 'abstract of final study report',

        'final study addendum report with abstract ': 'final study report with abstract',
        
        'interim report': 'interim study report',
        'summary interim report': 'interim study report',
        
        'progress report': 'progress study report',
        
        'tables': 'result tables only',
        'figures and tables': 'result tables only',

        **{
            original: 'other'
            for original in [
                'Declaration of Interests',
                'table - sites',
                'list - centers',
                'table - centers',
                'table - primary centers',
                'CSR upload delay notification',
                'change of sponsorship letter',
                'signiture page'
            ]
        },

        **{
            original: 'result publication'
            for original in [
                'original research article',
                'review article',
                'systematic review article'
            ]
        }
    }
}

comparison = document_CP.replace(replace_map).compare(document_PR)
relevant_to_check = comparison
# [
#     ~(
#         (comparison.manual_document_type.self == 'interim study report') & 
#         (comparison.manual_document_type.other == 'progress study report')
#     )
# ]

# display(replace_map)
display(relevant_to_check)

We can now save the non-matching parts of the Dataframes:

In [ ]:
document_CP.loc[relevant_to_check.index].merge(
    document_PR.manual_document_type, 
    left_index=True, 
    right_index=True, 
    how='left', 
    suffixes=('_CP', '_PR')
).to_excel('differences_imposed_documents_classification.xlsx')